In [1]:
import pandas as pd
import numpy as np

### Timestamp

In [2]:
# Timestamp, DatetimeIndex, Period, PeriodIndex

In [3]:
pd.Timestamp('20/07/2026 13:01')

Timestamp('2026-07-20 13:01:00')

In [5]:
pd.Timestamp(2006, 4, 12)

Timestamp('2006-04-12 00:00:00')

In [6]:
pd.Timestamp(2006, 4, 12).isoweekday()

3

In [8]:
pd.Timestamp(2006, 4, 12, 12, 5, 58).second

58

### Period

In [9]:
# Span of time
pd.Period('1/2016')

Period('2016-01', 'M')

In [14]:
pd.Period('04/12/2006')

Period('2006-04-12', 'D')

In [11]:
# 5 months after 2016
pd.Period('1/2016') + 5

Period('2016-06', 'M')

In [13]:
# 2 days before bday
pd.Period('04/12/2006') - 2

Period('2006-04-10', 'D')

### DatetimeIndex and PeriodIndex

In [15]:
t1 = pd.Series(list('abc'), [pd.Timestamp('2016-09-01'), pd.Timestamp('2016-09-02'), 
                             pd.Timestamp('2016-09-03')])
t1

2016-09-01    a
2016-09-02    b
2016-09-03    c
dtype: str

In [16]:
t2 = pd.Series(list('def'), [pd.Period('2016-09'), pd.Period('2016-10'), 
                             pd.Period('2016-11')])
t2

2016-09    d
2016-10    e
2016-11    f
Freq: M, dtype: str

In [ ]:
# dayfirst = True

### Timedelta

In [23]:
# Difference in times
pd.Timestamp(2026, 7, 20) - pd.Timestamp(2006, 4, 12)

Timedelta('7404 days 00:00:00')

In [26]:
# Add time
pd.Timestamp(2006, 4, 12) + pd.Timedelta('10000D')

Timestamp('2033-08-28 00:00:00')

### Offset

In [28]:
# Used in business analysis
pd.Timestamp(2006, 4, 12).weekday()

2

In [29]:
pd.Timestamp(2006, 4, 12) + pd.offsets.Week()

Timestamp('2006-04-19 00:00:00')

In [30]:
pd.Timestamp(2006, 4, 12) + pd.offsets.MonthEnd()

Timestamp('2006-04-30 00:00:00')

### Working with dates in dataframe

In [31]:
# Using date_range
dates = pd.date_range('10-01-2016', periods=9, freq='2W-SUN')
dates

DatetimeIndex(['2016-10-02', '2016-10-16', '2016-10-30', '2016-11-13',
               '2016-11-27', '2016-12-11', '2016-12-25', '2017-01-08',
               '2017-01-22'],
              dtype='datetime64[us]', freq='2W-SUN')

In [32]:
# There are many other frequencies that you can specify. For example, you can do business day
pd.date_range('10-01-2016', periods=9, freq='B')

DatetimeIndex(['2016-10-03', '2016-10-04', '2016-10-05', '2016-10-06',
               '2016-10-07', '2016-10-10', '2016-10-11', '2016-10-12',
               '2016-10-13'],
              dtype='datetime64[us]', freq='B')

In [33]:
# Or you can do quarterly, with the quarter start in June
pd.date_range('04-01-2016', periods=12, freq='QS-JUN')

DatetimeIndex(['2016-06-01', '2016-09-01', '2016-12-01', '2017-03-01',
               '2017-06-01', '2017-09-01', '2017-12-01', '2018-03-01',
               '2018-06-01', '2018-09-01', '2018-12-01', '2019-03-01'],
              dtype='datetime64[us]', freq='QS-JUN')

In [34]:
# Now, let's go back to our weekly on Sunday example and create a DataFrame using these dates, and some random
# data, and see what we can do with it.

dates = pd.date_range('10-01-2016', periods=9, freq='2W-SUN')
df = pd.DataFrame({'Count 1': 100 + np.random.randint(-5, 10, 9).cumsum(),
                  'Count 2': 120 + np.random.randint(-5, 10, 9)}, index=dates)
df

,Count 1,Count 2
2016-10-02,99,118
2016-10-16,97,124
2016-10-30,103,125
2016-11-13,99,129
2016-11-27,104,118
2016-12-11,99,121
2016-12-25,97,120
2017-01-08,104,118
2017-01-22,106,120


In [41]:
# First, we can check what day of the week a specific date is. For example, here we can see that all the dates
# in our index are on a Sunday. Which matches the frequency that we set
df.index.weekday

Index([6, 6, 6, 6, 6, 6, 6, 6, 6], dtype='int32')

In [36]:
# We can also use diff() to find the difference between each date's value.
df.diff()

,Count 1,Count 2
2016-10-02,NaN,NaN
2016-10-16,-2.0,6.0
2016-10-30,6.0,1.0
2016-11-13,-4.0,4.0
2016-11-27,5.0,-11.0
2016-12-11,-5.0,3.0
2016-12-25,-2.0,-1.0
2017-01-08,7.0,-2.0
2017-01-22,2.0,2.0


In [42]:
# Suppose we want to know what the mean count is for each month in our DataFrame. We can do this using
# resample. Converting from a higher frequency from a lower frequency is called downsampling (we'll talk about
# this in a moment)
df.resample('ME').mean()

,Count 1,Count 2
2016-10-31,99.666667,122.333333
2016-11-30,101.500000,123.500000
2016-12-31,98.000000,120.500000
2017-01-31,105.000000,119.000000


In [44]:
# Now let's talk about datetime indexing and slicing, which is a wonderful feature of the pandas DataFrame.
# For instance, we can use partial string indexing to find values from a particular year,
df.loc['2017']

,Count 1,Count 2
2017-01-08,104,118
2017-01-22,106,120


In [45]:
# Or we can do it from a particular month
df.loc['2016-12']

,Count 1,Count 2
2016-12-11,99,121
2016-12-25,97,120


In [46]:
# Or we can even slice on a range of dates For example, here we only want the values from December 2016
# onwards.
df.loc['2016-12':]

,Count 1,Count 2
2016-12-11,99,121
2016-12-25,97,120
2017-01-08,104,118
2017-01-22,106,120


In [47]:
df.loc['2016']

,Count 1,Count 2
2016-10-02,99,118
2016-10-16,97,124
2016-10-30,103,125
2016-11-13,99,129
2016-11-27,104,118
2016-12-11,99,121
2016-12-25,97,120


In [48]:
df

,Count 1,Count 2
2016-10-02,99,118
2016-10-16,97,124
2016-10-30,103,125
2016-11-13,99,129
2016-11-27,104,118
2016-12-11,99,121
2016-12-25,97,120
2017-01-08,104,118
2017-01-22,106,120
